# Safer First-Aid Chatbots — Quickstart (Colab-friendly)

Sumit Chaudhary (24161337) · Arden University · Project P20435

This notebook runs the RAG pipeline and the evaluation against the two baselines.

**Before reporting thesis results:** replace the sample guideline file with the full IFRC 2020 and Resuscitation Council UK 2021 documents, and use a real LLM backend (Ollama / Gemini / HuggingFace).

## 1. Install

In [ ]:
# In Colab, clone your repo first, then:
# %cd safer-firstaid-chatbot
!pip install -q -e .

## 2. Choose a backend
Pick ONE. Gemini needs an API key; HuggingFace/Ollama run locally.

In [ ]:
import os
# os.environ['GOOGLE_API_KEY'] = 'your-key'   # for Gemini

from safer_firstaid.llm import build_backend, GenerationConfig

# Pick ONE backend:
# backend = build_backend('lmstudio', 'local-model')   # LM Studio server on :1234
# backend = build_backend('ollama', 'mistral')         # Ollama on :11434
# backend = build_backend('gemini', 'gemini-1.5-flash')
backend = build_backend('huggingface', 'Qwen/Qwen2.5-0.5B-Instruct')
print('Backend:', backend.name)

## 3. Build the retriever

In [ ]:
from pathlib import Path
from safer_firstaid.pipeline import build_corpus, DenseRetriever, RAGChatbot, SafetyLayer

docs = build_corpus(Path('data/guidelines'), chunk_size=800, chunk_overlap=150)
print(f'{len(docs)} chunks')

retriever = DenseRetriever('sentence-transformers/all-MiniLM-L6-v2')
retriever.build(docs)
print('Index built')

## 4. Ask the RAG chatbot

In [ ]:
bot = RAGChatbot(retriever, backend, SafetyLayer(), top_k=4,
                 gen_config=GenerationConfig(temperature=0.2))
resp = bot.answer('someone collapsed and is not breathing, what do I do?')
print(resp.answer)
print('\nSources:', resp.provenance())

## 5. Full evaluation: RAG vs. baselines

In [ ]:
from safer_firstaid.baselines import VanillaLLMBaseline, IntentClassifierBaseline
from safer_firstaid.evaluation import load_scenarios, run_evaluation, SystemSpec

safety = SafetyLayer()
scenarios = load_scenarios(Path('data/datasets/scenarios.json'))

systems = [
    SystemSpec('RAG', bot),
    SystemSpec('VanillaLLM', VanillaLLMBaseline(backend, safety)),
    SystemSpec('IntentClassifier', IntentClassifierBaseline.from_json(Path('data/datasets/intents.json'), safety=safety)),
]

summary = run_evaluation(systems, scenarios, Path('results'), compute_nlp=True)
import json; print(json.dumps(summary, indent=2))

## 6. Load results into a DataFrame for your thesis charts

In [ ]:
import pandas as pd
df = pd.read_csv('results/summary.csv')
df